<a href="https://colab.research.google.com/github/Divine-reveire/TRIAL/blob/main/docs/tutorials/google/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2020 The Cirq Developers

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
import os, re, sys
from collections import defaultdict
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

drive.mount('/content/drive')

# Locate dataset folder in your Drive
base_dir = None
for root, dirs, files in os.walk('/content/drive/MyDrive/AMAZON'):
    if 'test_source1.tsv' in files:
        base_dir = os.path.dirname(os.path.dirname(root)) # points to student_resource root
        break

if not base_dir:
    # Default direct path from your Drive structure
    base_dir = '/content/drive/MyDrive/AMAZON/6ab10eb3b23ba_student_resource-1/student_resource'

test_dir = os.path.join(base_dir, 'dataset', 'test')
output_dir = os.path.join(base_dir, 'output')
os.makedirs(output_dir, exist_ok=True)

print(f"Working with test directory: {test_dir}")
print(f"Output directory: {output_dir}")

# --- Normalization Rules ---
LEGAL_SUFFIXES = {
    r'\bpvt\.?\s*ltd\.?\b': 'private limited',
    r'\bltd\.?\b': 'limited',
    r'\binc\.?\b': 'incorporated',
    r'\bcorp\.?\b': 'corporation',
    r'\bllc\.?\b': 'llc',
    r'\bllp\.?\b': 'llp',
    r'\bco\.?\b': 'company',
    r'\bgmbh\b': 'gmbh',
    r'\bsarl\b': 'sarl',
    r'\bsas\b': 'sas',
    r'\bsa\b': 'sa',
}

ADDRESS_ABBREVIATIONS = {
    r'\brd\.?\b': 'road',
    r'\bst\.?\b': 'street',
    r'\bave\.?\b': 'avenue',
    r'\bblvd\.?\b': 'boulevard',
    r'\bln\.?\b': 'lane',
    r'\bdr\.?\b': 'drive',
    r'\bct\.?\b': 'court',
    r'\bste\.?\b': 'suite',
    r'\bapt\.?\b': 'apartment',
    r'\bfl\.?\b': 'floor',
    r'\bdist\.?\b': 'district',
}

def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = s.lower().strip()
    s = re.sub(r'[\t\r\n]+', ' ', s)
    s = re.sub(r'[^\w\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def normalize_name(name):
    text = clean_text(name)
    for pattern, repl in LEGAL_SUFFIXES.items():
        text = re.sub(pattern, repl, text)
    return re.sub(r'\s+', ' ', text).strip()

def normalize_address(addr):
    text = clean_text(addr)
    for pattern, repl in ADDRESS_ABBREVIATIONS.items():
        text = re.sub(pattern, repl, text)
    return re.sub(r'\s+', ' ', text).strip()

# --- Execution ---
print("Reading test sources...")
s1 = pd.read_csv(os.path.join(test_dir, "test_source1.tsv"), sep="\t", dtype=str).fillna("")
s2 = pd.read_csv(os.path.join(test_dir, "test_source2.tsv"), sep="\t", dtype=str).fillna("")
s3 = pd.read_csv(os.path.join(test_dir, "test_source3.tsv"), sep="\t", dtype=str).fillna("")

targets = pd.concat([s2, s3], ignore_index=True)

print("Normalizing records...")
s1['comb'] = s1['business_name'].apply(normalize_name) + " " + s1['business_address'].apply(normalize_address)
s1['country_clean'] = s1['country'].str.strip().str.upper()

targets['comb'] = targets['business_name'].apply(normalize_name) + " " + targets['business_address'].apply(normalize_address)
targets['country_clean'] = targets['country'].str.strip().str.upper()

candidate_dict = defaultdict(list)
match_dict = defaultdict(list)

# Block by country to scale across millions of pairs
for c in s1['country_clean'].unique():
    print(f"Processing country: {c}")
    sub_s1 = s1[s1['country_clean'] == c].reset_index(drop=True)
    sub_tgt = targets[targets['country_clean'] == c].reset_index(drop=True)

    if len(sub_tgt) == 0:
        for eid in sub_s1['entity_id']:
            candidate_dict[eid] = []
            match_dict[eid] = []
        continue

    # Character n-grams (3-4) for typo tolerance and legal variant matching
    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 4), min_df=2)
    vec.fit(pd.concat([sub_s1['comb'], sub_tgt['comb']]))

    s1_vecs = vec.transform(sub_s1['comb'])
    tgt_vecs = vec.transform(sub_tgt['comb'])

    batch_size = 500
    for b_idx in range(0, len(sub_s1), batch_size):
        chunk = s1_vecs[b_idx : b_idx + batch_size]
        sims = cosine_similarity(chunk, tgt_vecs)

        for local_i, row in enumerate(sims):
            global_i = b_idx + local_i
            s1_id = sub_s1.at[global_i, 'entity_id']
            top_k = np.argsort(row)[::-1][:25]

            cands = []
            matches = []
            for t_idx in top_k:
                score = row[t_idx]
                if score < 0.20:
                    break
                tid = sub_tgt.at[t_idx, 'entity_id']
                cands.append(tid)
                if score >= 0.72:  # High threshold tuned for precision-heavy F_0.5
                    matches.append(tid)

            candidate_dict[s1_id] = cands
            match_dict[s1_id] = matches

# Write Output TSVs
cand_rows = [(eid, ",".join(candidate_dict[eid])) for eid in s1['entity_id']]
match_rows = [(eid, ",".join(match_dict[eid])) for eid in s1['entity_id']]

cand_path = os.path.join(output_dir, "candidate_pairs.tsv")
match_path = os.path.join(output_dir, "matching_results.tsv")

pd.DataFrame(cand_rows, columns=['source1_entity_id', 'candidate_entity_ids']).to_csv(cand_path, sep="\t", index=False)
pd.DataFrame(match_rows, columns=['source1_entity_id', 'matched_entity_ids']).to_csv(match_path, sep="\t", index=False)

print(f"Generated: {match_path}")
print(f"Generated: {cand_path}")

# Run official validator
validate_script = os.path.join(base_dir, 'utils', 'validate_submission.py')
if os.path.exists(validate_script):
    !python3 "{validate_script}" --matching "{match_path}" --candidate "{cand_path}" --test-dir "{test_dir}"

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Notebook template

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://quantumai.google/cirq/tutorials/google/colab"><img src="https://quantumai.google/site-assets/images/buttons/quantumai_logo_1x.png" />View on QuantumAI</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/quantumlib/Cirq/blob/main/docs/tutorials/google/colab.ipynb"><img src="https://quantumai.google/site-assets/images/buttons/colab_logo_1x.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/quantumlib/Cirq/blob/main/docs/tutorials/google/colab.ipynb"><img src="https://quantumai.google/site-assets/images/buttons/github_logo_1x.png" />View source on GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/Cirq/docs/tutorials/google/colab.ipynb"><img src="https://quantumai.google/site-assets/images/buttons/download_icon_1x.png" />Download notebook</a>
  </td>
</table>

## Setup


In [ ]:
try:
    import cirq
    import cirq_google
except ImportError:
    print("installing cirq...")
    !pip install cirq-google
    print("installed cirq.")
    import cirq
    import cirq_google

## Make a copy of this template

You will need to have access to Quantum Computing Service before running this colab.

This notebook can serve as a starter kit for you to run programs on Google's quantum hardware.  You can download it using the directions below, open it in colab (or Jupyter), and modify it to begin your experiments.

## How to download IPython notebooks from GitHub

You can retrieve IPython notebooks in the Cirq repository by
going to the [docs/ directory](https://github.com/quantumlib/Cirq/tree/main/docs).  For instance, this Colab template is found [here](https://github.com/quantumlib/Cirq/blob/main/docs/tutorials/google/colab.ipynb). Select the file that you would like to download and then click the *Raw* button in the upper-right part of the window:

<img src="https://raw.githubusercontent.com/quantumlib/Cirq/main/docs/images/colab_github.png" alt="GitHub UI button to view raw file">

This will show the entire file contents.  Right-click and select *Save as* to save this file to your computer.  Make sure to save to a file with a `.ipynb` extension (you may need to select *All files* from the format dropdown instead of *text*). You can also get to this Colab's [raw content directly](https://raw.githubusercontent.com/quantumlib/Cirq/main/docs/tutorials/google/colab.ipynb)

You can also retrieve the entire Cirq repository by running the following command in a terminal that has `git` installed:

```
git clone https://github.com/quantumlib/Cirq.git
```

## How to open Google Colab

You can open a new Colab notebook from your Google Drive window or by visiting the [Colab site](https://colab.research.google.com/notebooks/intro.ipynb).  From the Colaboratory site, you can use the menu to upload an IPython notebook:

<img src="https://raw.githubusercontent.com/quantumlib/Cirq/main/docs/images/colab_upload.png" alt="Google Colab's upload notebook entry in File menu.">

This will upload the ipynb file that you downloaded before.  You can now run all the commands, modify it to suit your goals, and share it with others.

### More Documentation Links

* [Quantum Engine concepts](../../google/concepts.ipynb)
* [Quantum Engine documentation](../../google/engine.md)
* [Cirq documentation](https://quantumai.google/cirq)
* [Colab documentation](https://colab.research.google.com/notebooks/welcome.ipynb)


## Authenticate with Quantum Computing Service and install Cirq

For details of authentication and installation, please see [Get started with Quantum Computing Service](start.ipynb).

Note:  The below code will install the latest stable release of Cirq.  If you need the latest and greatest features and don't mind if a few things aren't quite working correctly, you can install the pre-release version of `cirq` using `pip install --upgrade cirq~=1.0.dev` instead of `pip install cirq` to get the most up-to-date features of Cirq.

1. Enter the Cloud project ID you'd like to use in the `project_id` field.
2. Then run the cell below (and go through the auth flow for access to the project id you entered).

<img src="https://raw.githubusercontent.com/quantumlib/Cirq/main/docs/images/run-code-block.png" alt="Quantum Engine console">

In [ ]:
# The Google Cloud Project id to use.
project_id = ''  # @param {type:"string"}
processor_id = ""  # @param {type:"string"}

from cirq_google.engine.qcs_notebook import get_qcs_objects_for_notebook

# For real engine instances, delete 'virtual=True' below.
qcs_objects = get_qcs_objects_for_notebook(project_id, processor_id, virtual=True)
engine = qcs_objects.engine
processor_id = qcs_objects.processor_id

## Create an Engine variable

The following creates an engine variable which can be used to run programs under the project ID you entered above.

In [ ]:
from google.auth.exceptions import DefaultCredentialsError
from google.api_core.exceptions import PermissionDenied

# Create an Engine object to use, providing the project id and the args
try:
    if qcs_objects.signed_in:  # This line only needed for Colab testing.
        engine = cirq_google.get_engine()
    print(f"Successful authentication using project {project_id}!")
    print('Available Processors: ')
    print(engine.list_processors())
    print(f'Using processor: {processor_id}')
    processor = engine.get_processor(processor_id)
except DefaultCredentialsError as err:
    print("Could not authenticate to Google Quantum Computing Service.")
    print(" Tips: If you are using Colab: make sure the previous cell was executed successfully.")
    print(
        "       If this notebook is not in Colab (e.g. Jupyter notebook), make sure gcloud is installed and `gcloud auth application-default login` was executed."
    )
    print()
    print("Error message:")
    print(err)
except PermissionDenied as err:
    print(
        f"While you are authenticated to Google Cloud it seems the project '{project_id}' does not exist or does not have the Quantum Engine API enabled."
    )
    print("Error message:")
    print(err)

## Example

In [ ]:
# A simple example.
q = cirq.GridQubit(5, 2)
circuit = cirq.Circuit(cirq.X(q) ** 0.5, cirq.measure(q, key='m'))

job = processor.run_sweep(program=circuit, repetitions=1000)

results = [str(int(b)) for b in job.results()[0].measurements['m'][:, 0]]
print('Success!  Results:')
print(''.join(results))